# Chapter 13: Quadratic Forms

Source span: printed Volume II pages 86-115; PDF pages 95-124. This notebook is an original, visualization-first lesson on quadratic forms over the real numbers. The goal is not to memorize a classification table, but to learn how a quadratic expression carves space into visible regions and how changes of basis reveal the form's stable invariants.

The central question for the chapter is this: when two formulas look different, how can we tell whether they describe the same geometry? For quadratic forms the answer is mostly controlled by three counts: how many independent directions are positive, how many are negative, and how many are invisible to the form. Those counts are the signature and the radical dimension. They decide the local pictures: ellipses, cones, hyperboloids, flat directions, and the hyperbolic planes that appear in Witt decompositions.

## Translation guide

| Book concept | Computational model in this notebook | Visual cue |
| --- | --- | --- |
| Quadratic form `q` on a real vector space `V` | a symmetric matrix `A` with `q(x) = x.T @ A @ x` | level sets of `q` |
| Polar bilinear form `B` | `B(x, y) = x.T @ A @ y` | orthogonal directions satisfy `B(x, y) = 0` |
| Change of basis | congruence `A -> P.T @ A @ P` | the picture is re-coordinatized, not reclassified |
| Signature | counts of positive, negative, and zero eigenvalues | ellipsoid / saddle / cone / flat direction |
| Isotropic vector | nonzero `x` with `q(x) = 0` | lines or cones through the origin |
| Radical | `ker(A)`, equivalently all vectors orthogonal to every vector | a direction the form cannot detect |
| Hyperbolic plane | matrix `[[0, 1], [1, 0]]`, with `q(x, y) = 2xy` | two isotropic axes paired together |
| Reflection | `x -> x - 2 B(x,v)/B(v,v) v` for anisotropic `v` | form-preserving mirror move |

The matrix model is not a restriction: every finite-dimensional real quadratic form becomes a symmetric matrix after a basis is chosen. What changes under a new basis is the matrix, while the signature and radical stay fixed.

## Route through the chapter

1. Build the form from a matrix and recover its polar form.
2. Read the signature from level sets in dimension two.
3. Track isotropic cones and radicals as visible zero geometry.
4. Diagonalize by orthogonal coordinates and interpret the law of inertia.
5. Split indefinite spaces into hyperbolic planes plus anisotropic leftovers: the Witt viewpoint.
6. Use reflections as concrete form-preserving transformations.
7. Finish with executable checks that the algebra and generated artifacts agree.

## Visualization storyboard

- A four-panel contour board compares positive, negative, indefinite, and degenerate forms.
- A 3D interactive Plotly scene shows the Lorentz cone `x^2 + y^2 - z^2 = 0` together with nearby level surfaces.
- A heatmap pair displays an arbitrary Gram matrix before and after diagonalization.
- A reflection diagram in the Minkowski plane shows how products of reflections move points along hyperbolas while preserving the cone.
- A compact CSV table records signatures for reusable examples, and a JSON check file records the numeric invariants verified at the end.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def discover_book_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
            return candidate
    raise RuntimeError("Could not locate the Geometry II book root")


BOOK_ROOT = discover_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-13"
for kind in ["figures", "plots", "tables", "checks"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)

import math
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import Markdown, display

from utils.artifacts import artifact_path, assert_artifacts, display_artifact, save_csv, save_json, save_matplotlib, save_plotly_html
from utils.geometry import quadratic_signature
from utils.plotting import COLORS

np.set_printoptions(precision=4, suppress=True)


def rel(path: Path) -> str:
    return Path(path).resolve().relative_to(BOOK_ROOT.resolve()).as_posix()


print(f"BOOK_ROOT = {BOOK_ROOT.name}")
print(f"Artifacts will be written under {rel(ARTIFACT_ROOT)}")

## 1. From a formula to a bilinear form

A quadratic form is homogeneous of degree two: scaling a vector by `t` scales the value by `t^2`. In coordinates it is a sum of squared terms and cross terms. Cross terms are the first place where the matrix language pays for itself: the coefficient of `x_i x_j` is split symmetrically between the two off-diagonal matrix entries.

The polar form is the symmetric bilinear measurement hiding inside the quadratic one. For `q(x) = x.T A x` with `A` symmetric, the associated bilinear form is `B(x, y) = x.T A y`. It can be recovered from `q` alone by the identity

`2 B(x, y) = q(x + y) - q(x) - q(y)`.

This identity is the computational bridge between the visible level sets of `q` and the orthogonality relation used throughout the chapter.

In [ ]:
def symmetrize(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=float)
    return 0.5 * (matrix + matrix.T)


def q_value(A: np.ndarray, x: np.ndarray) -> float:
    A = symmetrize(A)
    x = np.asarray(x, dtype=float)
    return float(x.T @ A @ x)


def polar(A: np.ndarray, x: np.ndarray, y: np.ndarray) -> float:
    A = symmetrize(A)
    return float(np.asarray(x, dtype=float).T @ A @ np.asarray(y, dtype=float))


def radical_dimension(A: np.ndarray, *, tol: float = 1e-9) -> int:
    eig = np.linalg.eigvalsh(symmetrize(A))
    return int(np.sum(np.abs(eig) <= tol))


def has_isotropic_vectors(A: np.ndarray, *, tol: float = 1e-9) -> bool:
    p, n, z = quadratic_signature(symmetrize(A), tol=tol)
    return z > 0 or (p > 0 and n > 0)


A_demo = np.array([[2.0, -0.5, 0.0], [-0.5, 1.0, 0.75], [0.0, 0.75, -1.0]])
x_demo = np.array([1.0, 2.0, -1.0])
y_demo = np.array([-0.25, 1.5, 0.5])
polarization_error = q_value(A_demo, x_demo + y_demo) - q_value(A_demo, x_demo) - q_value(A_demo, y_demo) - 2 * polar(A_demo, x_demo, y_demo)
print("signature(A_demo) =", quadratic_signature(A_demo))
print("polarization error =", polarization_error)

## 2. Signature is a visible invariant

In two variables the signature can be read almost by sight. A positive definite form has nested ellipses around the origin. A negative definite form has the same level sets with signs reversed. An indefinite form has two families of hyperbolas and a nontrivial zero set, the pair of isotropic lines. A degenerate form forgets at least one direction; its level sets stretch into parallel bands, and the zero set contains the radical or a larger isotropic set.

The next figure uses the same coordinate window for four forms. The red zero set is the feature to watch: in definite cases it collapses to the origin; in indefinite cases it becomes a cone; in degenerate cases it exposes directions that the form cannot measure.

In [ ]:
forms_2d = [
    ("positive definite", np.array([[2.0, 0.55], [0.55, 1.0]])),
    ("negative definite", -np.array([[1.25, -0.35], [-0.35, 1.8]])),
    ("indefinite", np.array([[1.0, 0.0], [0.0, -1.0]])),
    ("rank-one semidefinite", np.array([[1.0, 0.0], [0.0, 0.0]])),
]

xs = np.linspace(-2.25, 2.25, 401)
ys = np.linspace(-2.25, 2.25, 401)
X, Y = np.meshgrid(xs, ys)
points = np.stack([X, Y], axis=-1)
base_levels = [-6, -3, -1, -0.25, 0.25, 1, 3, 6]

fig, axes = plt.subplots(2, 2, figsize=(10.5, 9.0), constrained_layout=True)
for ax, (name, A) in zip(axes.ravel(), forms_2d):
    Z = np.einsum("...i,ij,...j->...", points, A, points)
    levels = [level for level in base_levels if Z.min() < level < Z.max()]
    if levels:
        contour = ax.contour(X, Y, Z, levels=levels, cmap="coolwarm", linewidths=1.2)
        ax.clabel(contour, inline=True, fontsize=7, fmt="%g")
    p, n, z = quadratic_signature(A)
    ax.set_title(f"{name}: signature {(p, n, z)}", loc="left", fontsize=11, fontweight="bold")
    ax.axhline(0, color="#cbd5e1", linewidth=0.9)
    ax.axvline(0, color="#cbd5e1", linewidth=0.9)
    if name == "positive definite" or name == "negative definite":
        ax.scatter([0], [0], s=70, color=COLORS["red"], edgecolor="white", zorder=5, label="q=0")
    elif name == "indefinite":
        ax.plot(xs, xs, color=COLORS["red"], linewidth=2.4, label="q=0")
        ax.plot(xs, -xs, color=COLORS["red"], linewidth=2.4)
    else:
        ax.axvline(0, color=COLORS["red"], linewidth=2.4, label="q=0")
    ax.set_xlim(xs[0], xs[-1])
    ax.set_ylim(ys[0], ys[-1])
    ax.set_aspect("equal")
    ax.grid(True, color="#e5e7eb", linewidth=0.8)
    ax.tick_params(labelsize=8)
    ax.legend(loc="upper right", fontsize=8, frameon=True)

signature_path = artifact_path(13, "figures", "signature-slices.png", book_root=BOOK_ROOT)
save_matplotlib(fig, signature_path, dpi=170)
plt.close(fig)
display_artifact(signature_path, width=880)

The red set is already the beginning of the classification. A nonzero isotropic vector can exist in two ways. The first is degeneracy: a vector in the radical satisfies `B(v, w) = 0` for every `w`, hence `q(v) = 0`. The second is cancellation: a positive direction and a negative direction combine to make `q` vanish without either direction being invisible. That cancellation is why indefinite forms have cones.

This distinction matters later in projective geometry. A cone coming from a nondegenerate indefinite form carries real incidence geometry; a radical points to a quotient space that must be taken before the form becomes nondegenerate.

In [ ]:
example_matrices = {
    "positive plane": np.array([[2.0, 0.35], [0.35, 1.0]]),
    "negative plane": np.array([[-2.0, 0.2], [0.2, -0.75]]),
    "hyperbolic plane": np.array([[0.0, 1.0], [1.0, 0.0]]),
    "Lorentz 3-space": np.diag([1.0, 1.0, -1.0]),
    "radical line attached": np.diag([2.0, -1.0, 0.0]),
    "rank-one form": np.diag([1.0, 0.0, 0.0]),
}

rows = []
for name, A in example_matrices.items():
    eig = np.linalg.eigvalsh(symmetrize(A))
    p, n, z = quadratic_signature(A)
    rows.append(
        {
            "name": name,
            "dimension": A.shape[0],
            "positive": p,
            "negative": n,
            "zero": z,
            "rank": int(np.linalg.matrix_rank(A, tol=1e-9)),
            "has_isotropic_vectors": has_isotropic_vectors(A),
            "eigenvalues": ";".join(f"{value:.6g}" for value in eig),
        }
    )

signature_table_path = artifact_path(13, "tables", "signature-examples.csv", book_root=BOOK_ROOT)
save_csv(rows, signature_table_path)
display(Markdown(f"Signature table written to `{rel(signature_table_path)}`."))
display_artifact(signature_table_path)

## 3. The Lorentz cone as a model for isotropy

The form `q(x, y, z) = x^2 + y^2 - z^2` is the most economical three-dimensional model for a nondegenerate indefinite form. Its zero set is a double cone. Points inside one region have negative value, points outside have positive value, and the cone itself is made of nonzero isotropic directions.

The interactive scene saves as HTML so the cone can be rotated. The gray cone is `q = 0`. The blue one-sheet surface is `q = 1`, and the orange two-sheet surface is `q = -1`. These three objects are not separate topics; they are one quadratic form seen at three levels.

In [ ]:
r = np.linspace(0.0, 2.2, 72)
theta = np.linspace(0.0, 2 * np.pi, 96)
R, T = np.meshgrid(r, theta)
Xc = R * np.cos(T)
Yc = R * np.sin(T)
Zc = R

r_pos = np.linspace(1.0, 2.2, 62)
Rp, Tp = np.meshgrid(r_pos, theta)
Xp = Rp * np.cos(Tp)
Yp = Rp * np.sin(Tp)
Zp = np.sqrt(np.maximum(Rp**2 - 1.0, 0.0))

r_neg = np.linspace(0.0, 1.6, 58)
Rn, Tn = np.meshgrid(r_neg, theta)
Xn = Rn * np.cos(Tn)
Yn = Rn * np.sin(Tn)
Zn = np.sqrt(1.0 + Rn**2)

fig3d = go.Figure()
for sign in [1, -1]:
    fig3d.add_trace(
        go.Surface(
            x=Xc,
            y=Yc,
            z=sign * Zc,
            colorscale=[[0, "rgba(120,120,120,0.18)"], [1, "rgba(120,120,120,0.18)"]],
            showscale=False,
            opacity=0.28,
            name="q=0 cone",
        )
    )
for sign in [1, -1]:
    fig3d.add_trace(
        go.Surface(
            x=Xp,
            y=Yp,
            z=sign * Zp,
            colorscale="Blues",
            showscale=False,
            opacity=0.62,
            name="q=1",
        )
    )
for sign in [1, -1]:
    fig3d.add_trace(
        go.Surface(
            x=Xn,
            y=Yn,
            z=sign * Zn,
            colorscale="Oranges",
            showscale=False,
            opacity=0.72,
            name="q=-1",
        )
    )
fig3d.update_layout(
    title="Lorentz form: x^2 + y^2 - z^2",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="cube",
        camera=dict(eye=dict(x=1.5, y=-1.6, z=1.1)),
    ),
    margin=dict(l=0, r=0, t=45, b=0),
    height=640,
)
cone_path = artifact_path(13, "plots", "lorentz-cone-levels.html", book_root=BOOK_ROOT)
save_plotly_html(fig3d, cone_path)
display_artifact(cone_path, width=880, height=680)

## 4. Diagonalization and the law of inertia

A symmetric real matrix can be diagonalized by an orthonormal matrix, but the deeper quadratic-form fact is about congruence, not similarity. Under a basis change `x = P y`, the matrix becomes `P.T A P`. The eigenvalues themselves can move under general congruence, yet the number of positive, negative, and zero diagonal entries after reduction cannot change. That is Sylvester's law of inertia.

The heatmaps below use a deliberately mixed Gram matrix. The left panel is hard to read because every coordinate talks to every other coordinate. The right panel is a diagonal coordinate system: positive entries, negative entries, and a zero radical direction are separated. The picture is the computational version of orthogonalization.

In [ ]:
D_seed = np.diag([3.0, 1.2, -2.0, 0.0])
P_seed = np.array(
    [
        [1.0, 0.25, -0.35, 0.8],
        [0.15, 1.0, 0.5, -0.25],
        [-0.4, 0.2, 1.0, 0.45],
        [0.3, -0.55, 0.1, 1.0],
    ]
)
A_mixed = symmetrize(P_seed.T @ D_seed @ P_seed)
evals, U = np.linalg.eigh(A_mixed)
A_diag = U.T @ A_mixed @ U

max_abs = max(np.max(np.abs(A_mixed)), np.max(np.abs(A_diag)))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.6), constrained_layout=True)
for ax, matrix, title in [
    (axes[0], A_mixed, f"mixed Gram matrix {quadratic_signature(A_mixed)}"),
    (axes[1], A_diag, f"orthogonal coordinates {quadratic_signature(A_diag)}"),
]:
    image = ax.imshow(matrix, cmap="coolwarm", vmin=-max_abs, vmax=max_abs)
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold")
    ax.set_xticks(range(matrix.shape[0]))
    ax.set_yticks(range(matrix.shape[0]))
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            color = "white" if abs(value) > 0.45 * max_abs else COLORS["ink"]
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8, color=color)
fig.colorbar(image, ax=axes, shrink=0.82, label="entry value")
diagonalization_path = artifact_path(13, "figures", "orthogonalization-heatmap.png", book_root=BOOK_ROOT)
save_matplotlib(fig, diagonalization_path, dpi=170)
plt.close(fig)
display_artifact(diagonalization_path, width=880)

The zero diagonal entry on the right is not a numerical accident. It records the radical. Algebraically, the radical is the nullspace of the Gram matrix. Geometrically, it is the direction along which all pairings vanish. If a form has a radical, classification usually begins by splitting it off or passing to the quotient by it. Only then does the nondegenerate part carry the full positive-versus-negative geometry.

In [ ]:
def nullspace(A: np.ndarray, *, tol: float = 1e-9) -> np.ndarray:
    u, s, vh = np.linalg.svd(symmetrize(A))
    mask = s <= tol
    if not np.any(mask):
        return np.zeros((A.shape[0], 0))
    return vh[mask].T


rad_basis = nullspace(A_mixed)
print("eigenvalues after orthogonalization:", evals)
print("radical dimension from eigenvalues:", radical_dimension(A_mixed))
print("radical basis shape:", rad_basis.shape)
if rad_basis.shape[1]:
    print("A @ radical basis residual:", np.linalg.norm(A_mixed @ rad_basis))

## 5. Witt decomposition: pairing isotropic directions

Indefinite geometry is clearest when we stop treating isotropic directions as defects. In the hyperbolic plane with matrix

`H = [[0, 1], [1, 0]]`,

both coordinate axes are isotropic because `q(x, y) = 2xy`. They are not radical directions: each axis pairs nontrivially with the other. Replacing the isotropic basis `(e, f)` by `(e + f, e - f)` turns the same form into a positive square and a negative square:

`(e + f).T H (e + f) = 2`, and `(e - f).T H (e - f) = -2`.

A Witt decomposition repeats this idea. It pairs as many positive and negative directions as possible into hyperbolic planes, leaves an anisotropic definite part if one sign count is larger, and carries along the radical if the original form is degenerate. The number of hyperbolic planes is the Witt index, `min(p, n)`, for a real nondegenerate form of signature `(p, n, 0)`.

In [ ]:
H = np.array([[0.0, 1.0], [1.0, 0.0]])
change_from_isotropic = np.array([[1.0, 1.0], [1.0, -1.0]])
H_diagonal = change_from_isotropic.T @ H @ change_from_isotropic

witt_examples = []
for name, A in example_matrices.items():
    p, n, z = quadratic_signature(A)
    witt_examples.append(
        {
            "name": name,
            "signature": (p, n, z),
            "witt_index": min(p, n),
            "anisotropic_dimension": abs(p - n),
            "radical_dimension": z,
        }
    )

print("H in isotropic basis:")
print(H)
print("H in the e+f, e-f basis:")
print(H_diagonal)
print("Witt bookkeeping:")
for row in witt_examples:
    print(row)

## 6. Reflections preserve the quadratic form

Whenever `v` is anisotropic, meaning `q(v) != 0`, the formula

`s_v(x) = x - 2 B(x, v) / B(v, v) v`

is a reflection for the quadratic form. It sends `v` to `-v`, fixes the hyperplane `B(x, v) = 0`, and preserves `q`. In Euclidean signature this is the familiar mirror reflection. In indefinite signature it can look less familiar, but the same calculation works.

The next figure works in the Minkowski plane `q(x, y) = x^2 - y^2`. The dashed diagonals are isotropic lines. The product of two reflections sends the right branch of `q = 1` to itself and moves points along that hyperbola, the indefinite analogue of composing two Euclidean mirror reflections to get a rotation.

In [ ]:
def reflection_matrix(A: np.ndarray, v: np.ndarray) -> np.ndarray:
    A = symmetrize(A)
    v = np.asarray(v, dtype=float).reshape(-1, 1)
    denom = float(v.T @ A @ v)
    if abs(denom) <= 1e-12:
        raise ValueError("reflection vector must be anisotropic")
    return np.eye(A.shape[0]) - 2.0 * (v @ (v.T @ A)) / denom


A_minkowski = np.diag([1.0, -1.0])
rapidity = 0.42
v1 = np.array([1.0, 0.0])
v2 = np.array([math.cosh(rapidity), math.sinh(rapidity)])
S1 = reflection_matrix(A_minkowski, v1)
S2 = reflection_matrix(A_minkowski, v2)
boost_from_reflections = S2 @ S1

t = np.linspace(-1.2, 1.2, 15)
right_branch = np.column_stack([np.cosh(t), np.sinh(t)])
moved = (boost_from_reflections @ right_branch.T).T

fig, ax = plt.subplots(figsize=(7.2, 6.0), constrained_layout=True)
span = np.linspace(-2.3, 2.3, 250)
ax.plot(span, span, linestyle="--", color=COLORS["gray"], linewidth=1.4, label="q=0")
ax.plot(span, -span, linestyle="--", color=COLORS["gray"], linewidth=1.4)
for sign in [1, -1]:
    tt = np.linspace(-1.45, 1.45, 300)
    branch = np.column_stack([sign * np.cosh(tt), np.sinh(tt)])
    ax.plot(branch[:, 0], branch[:, 1], color=COLORS["ink"], linewidth=1.9)
ax.scatter(right_branch[:, 0], right_branch[:, 1], s=30, color=COLORS["blue"], label="before")
ax.scatter(moved[:, 0], moved[:, 1], s=30, color=COLORS["orange"], label="after two reflections")
for start, end in zip(right_branch[::2], moved[::2]):
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", color=COLORS["teal"], lw=1.2, shrinkA=2, shrinkB=2))
ax.quiver([0, 0], [0, 0], [v1[0], v2[0]], [v1[1], v2[1]], angles="xy", scale_units="xy", scale=1, color=[COLORS["red"], COLORS["purple"]], width=0.007)
ax.text(v1[0] + 0.06, v1[1] + 0.04, "v1", color=COLORS["red"], fontsize=10)
ax.text(v2[0] + 0.06, v2[1] + 0.04, "v2", color=COLORS["purple"], fontsize=10)
ax.set_title("Minkowski reflections preserve q=x^2-y^2", loc="left", fontsize=12, fontweight="bold")
ax.set_xlim(-2.3, 2.3)
ax.set_ylim(-2.3, 2.3)
ax.set_aspect("equal")
ax.grid(True, color="#e5e7eb", linewidth=0.8)
ax.legend(loc="upper left", fontsize=8)
reflection_path = artifact_path(13, "figures", "reflection-factorization.png", book_root=BOOK_ROOT)
save_matplotlib(fig, reflection_path, dpi=170)
plt.close(fig)
display_artifact(reflection_path, width=760)

## Applied lab: classifying a quadratic form by computation

When a new quadratic form appears, the workflow is short and robust.

1. Symmetrize the matrix; only the symmetric part contributes to `x.T A x`.
2. Compute the signature and rank.
3. Locate the radical by solving `A x = 0`.
4. If both positive and negative signs occur, inspect the isotropic cone; otherwise expect only the origin to solve `q(x) = 0` unless the form is degenerate.
5. Use congruence or an orthogonal diagonalization to expose the positive, negative, and zero directions.
6. For transformations, test the matrix identity `T.T A T = A` rather than checking a few sample points.

The point of the lab is not to replace proof with numerics. It is to make the proof objects visible enough that the invariants feel inevitable.

In [ ]:
def classify_quadratic_form(A: np.ndarray, *, tol: float = 1e-9) -> dict[str, object]:
    A = symmetrize(A)
    eig = np.linalg.eigvalsh(A)
    p, n, z = quadratic_signature(A, tol=tol)
    return {
        "dimension": A.shape[0],
        "signature": (p, n, z),
        "rank": int(np.sum(np.abs(eig) > tol)),
        "radical_dimension": int(np.sum(np.abs(eig) <= tol)),
        "witt_index": min(p, n),
        "has_isotropic_vectors": bool(z > 0 or (p > 0 and n > 0)),
        "eigenvalues": eig,
    }


lab_form = np.array(
    [
        [0.0, 2.0, 0.0, 0.0],
        [2.0, 0.0, 1.0, 0.0],
        [0.0, 1.0, 3.0, 0.0],
        [0.0, 0.0, 0.0, 0.0],
    ]
)
classification = classify_quadratic_form(lab_form)
classification

## Final sanity checks

The checks below turn the chapter's main claims into executable assertions. They verify polarization, congruence invariance of signature, hyperbolic-plane diagonalization, radical detection, reflection preservation, and the existence of every artifact generated by the notebook.

In [ ]:
rng = np.random.default_rng(1307)

# Polarization identity on random samples.
polarization_errors = []
for _ in range(20):
    x = rng.normal(size=3)
    y = rng.normal(size=3)
    err = q_value(A_demo, x + y) - q_value(A_demo, x) - q_value(A_demo, y) - 2.0 * polar(A_demo, x, y)
    polarization_errors.append(abs(err))

# Congruence invariance of signature.
base = np.diag([4.0, 1.0, -2.5, 0.0])
P = np.array(
    [
        [1.0, 0.4, -0.2, 0.3],
        [0.2, 1.0, 0.35, -0.4],
        [-0.1, 0.25, 1.0, 0.5],
        [0.3, -0.2, 0.15, 1.0],
    ]
)
congruent = P.T @ base @ P
signature_invariance_ok = quadratic_signature(base) == quadratic_signature(congruent)

# Hyperbolic plane diagonalization.
hyperbolic_target = np.diag([2.0, -2.0])
hyperbolic_error = float(np.linalg.norm(H_diagonal - hyperbolic_target))

# Radical detection.
radical_residual = float(np.linalg.norm(A_mixed @ rad_basis)) if rad_basis.shape[1] else 0.0
radical_dimension_ok = radical_dimension(A_mixed) == rad_basis.shape[1]

# Reflection preservation.
reflection_errors = []
for T in [S1, S2, boost_from_reflections]:
    reflection_errors.append(float(np.linalg.norm(T.T @ A_minkowski @ T - A_minkowski)))
for point in rng.normal(size=(20, 2)):
    moved_point = boost_from_reflections @ point
    reflection_errors.append(abs(q_value(A_minkowski, moved_point) - q_value(A_minkowski, point)))

artifact_files = [
    signature_path,
    signature_table_path,
    cone_path,
    diagonalization_path,
    reflection_path,
]
assert_artifacts(artifact_files, min_bytes=64)

final_sanity = {
    "max_polarization_error": float(max(polarization_errors)),
    "signature_invariance_ok": bool(signature_invariance_ok),
    "hyperbolic_plane_diagonalization_error": hyperbolic_error,
    "radical_residual": radical_residual,
    "radical_dimension_ok": bool(radical_dimension_ok),
    "max_reflection_preservation_error": float(max(reflection_errors)),
    "artifact_paths": [rel(path) for path in artifact_files],
}

assert final_sanity["max_polarization_error"] < 1e-10
assert final_sanity["signature_invariance_ok"]
assert final_sanity["hyperbolic_plane_diagonalization_error"] < 1e-12
assert final_sanity["radical_residual"] < 1e-8
assert final_sanity["radical_dimension_ok"]
assert final_sanity["max_reflection_preservation_error"] < 1e-10

final_sanity_path = artifact_path(13, "checks", "final-sanity.json", book_root=BOOK_ROOT)
save_json(final_sanity, final_sanity_path)
assert_artifacts([*artifact_files, final_sanity_path], min_bytes=64)

display(Markdown(f"Final sanity checks written to `{rel(final_sanity_path)}`."))
final_sanity

## Takeaways

Quadratic forms are geometry in a compressed algebraic package. The matrix entries are coordinate-dependent, but the signature is not. Once a basis exposes the positive, negative, and radical directions, the visible behavior of the form becomes almost automatic.

Isotropic vectors are the main warning sign that Euclidean intuition is too narrow. In a degenerate form they can come from directions the form ignores; in a nondegenerate indefinite form they come from cancellation between positive and negative directions. The hyperbolic plane is the smallest model where that cancellation is structural rather than accidental.

Reflections close the chapter's circle. They are defined from the polar form, preserve the quadratic form, and generate many of the natural symmetries of nondegenerate quadratic spaces. The final JSON checks record these facts numerically, while the artifacts provide the visual handles: contour slices, the Lorentz cone, diagonalization, and reflection motion.